# StageBridge Quickstart Tutorial

This notebook demonstrates the basic workflow for using StageBridge to model cell state transitions in spatial transcriptomics data.

**What you'll learn:**
1. Loading a pretrained model
2. Preparing spatial neighborhoods
3. Computing niche-aware embeddings
4. Predicting cell state transitions
5. Visualizing results

**Requirements:**
```bash
pip install stagebridge scanpy anndata matplotlib
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad

import stagebridge as sb

# Set random seed for reproducibility
np.random.seed(42)

print(f"StageBridge version: {sb.__version__}")

## 1. Create Synthetic Data (for demonstration)

In practice, you would load your own spatial transcriptomics data. Here we create synthetic data to demonstrate the API.

In [ ]:
def create_synthetic_spatial_data(n_cells=1000):
    """Create synthetic spatial transcriptomics data.
    
    Simulates a tissue section with:
    - Spatial gradient from Normal (left) to Invasive (right)
    - scVI embeddings (40d latent)
    - HLCA and LuCA reference embeddings
    """
    # Spatial coordinates (1mm x 1mm tissue)
    coords = np.random.rand(n_cells, 2) * 1000
    
    # Stage labels based on spatial gradient
    x_normalized = coords[:, 0] / 1000
    stage_probs = np.column_stack([
        1 - x_normalized,           # Normal on left
        2 * np.abs(x_normalized - 0.5),  # Preinvasive in middle
        x_normalized,                # Invasive on right
    ])
    stage_idx = np.argmax(stage_probs + np.random.randn(n_cells, 3) * 0.2, axis=1)
    stages = np.array(["Normal", "Preinvasive", "Invasive"])[stage_idx]
    
    # Generate embeddings with stage structure
    base_embedding = np.random.randn(n_cells, 40).astype(np.float32)
    stage_shift = np.zeros((n_cells, 40))
    stage_shift[stages == "Normal", :5] = -1
    stage_shift[stages == "Invasive", :5] = 1
    X_scvi = base_embedding + stage_shift
    
    # Reference embeddings
    X_hlca = np.random.randn(n_cells, 30).astype(np.float32)
    X_luca = np.random.randn(n_cells, 10).astype(np.float32)
    
    # Create AnnData
    adata = ad.AnnData(
        X=np.random.randn(n_cells, 2000).astype(np.float32),
        obs=pd.DataFrame({
            "stage": pd.Categorical(stages, categories=["Normal", "Preinvasive", "Invasive"]),
            "donor_id": np.random.choice(["D1", "D2", "D3"], n_cells),
            "cell_type": np.random.choice(["Epithelial", "T cell", "Macrophage", "Fibroblast"], n_cells),
        }),
        obsm={
            "X_scvi": X_scvi,
            "X_scANVI_hlca": X_hlca,
            "X_scVI_luca": X_luca,
            "spatial": coords,
        },
    )
    
    return adata

# Create data
adata = create_synthetic_spatial_data(n_cells=2000)
print(f"Created AnnData with {adata.n_obs} cells")
print(f"Stage distribution:\n{adata.obs['stage'].value_counts()}")

In [ ]:
# Visualize spatial distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot by stage
stage_colors = {"Normal": "#1B4F72", "Preinvasive": "#2E86AB", "Invasive": "#922B21"}
for stage in ["Normal", "Preinvasive", "Invasive"]:
    mask = adata.obs["stage"] == stage
    axes[0].scatter(
        adata.obsm["spatial"][mask, 0],
        adata.obsm["spatial"][mask, 1],
        c=stage_colors[stage],
        label=stage,
        alpha=0.6,
        s=10,
    )
axes[0].set_xlabel("X (μm)")
axes[0].set_ylabel("Y (μm)")
axes[0].set_title("Spatial Distribution by Stage")
axes[0].legend()

# Plot by cell type
sc.pl.embedding(adata, basis="spatial", color="cell_type", ax=axes[1], show=False, title="Cell Types")

plt.tight_layout()
plt.show()

## 2. Prepare Neighborhoods

StageBridge uses a receiver-centered niche model. For each cell (receiver), we identify its spatial neighbors organized into concentric rings.

In [ ]:
# Prepare receiver-centered neighborhoods
neighborhoods = sb.prepare_neighborhoods(
    adata,
    ring_radii=[50, 100, 150, 200],  # Ring boundaries in microns
    embedding_key="X_scvi",           # Cell embeddings from scVI
    hlca_key="X_scANVI_hlca",          # HLCA reference projection
    luca_key="X_scVI_luca",            # LuCA reference projection
    spatial_key="spatial",             # Spatial coordinates
    max_cells_per_ring=32,             # Max neighbors per ring
)

print(f"Prepared {len(neighborhoods)} neighborhoods")
print(f"Columns: {list(neighborhoods.columns)}")

## 3. Load Pretrained Model

Load a pretrained StageBridge model from a checkpoint.

In [ ]:
# Option 1: Load from checkpoint (use your trained model)
# model = sb.StageBridge.from_pretrained("runs/stagebridge/checkpoints/best.pt")

# Option 2: Create untrained model for demonstration
from stagebridge.models import StageBridge as StageBridgeModel, StageBridgeConfig

config = StageBridgeConfig(
    input_dim=40,
    hidden_dim=128,
    num_heads=4,
    num_encoder_layers=2,
    use_cross_attn_drift=True,
    use_context_refiner=True,
)

model_core = StageBridgeModel(config)
model = sb.StageBridge(model_core, config, device="cpu")

print(f"Model configuration:")
print(f"  Hidden dim: {model.config.hidden_dim}")
print(f"  Num heads: {model.config.num_heads}")
print(f"  Device: {model.device}")

## 4. Compute Niche Embeddings

Get niche-aware context embeddings that capture spatial relationships.

In [ ]:
# Compute niche embeddings
niche_output = model.embed_niches(
    neighborhoods,
    batch_size=256,
    return_tokens=True,  # Also return individual token embeddings
)

print(f"Niche embeddings shape: {niche_output.embeddings.shape}")
print(f"Context tokens shape: {niche_output.context_tokens.shape if niche_output.context_tokens is not None else 'None'}")
print(f"Attention weights shape: {niche_output.attention_weights.shape if niche_output.attention_weights is not None else 'None'}")

In [ ]:
# Store embeddings in AnnData
adata.obsm["X_niche"] = niche_output.embeddings

# Visualize with PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
niche_pca = pca.fit_transform(niche_output.embeddings)

fig, ax = plt.subplots(figsize=(8, 6))
for stage in ["Normal", "Preinvasive", "Invasive"]:
    mask = adata.obs["stage"] == stage
    ax.scatter(
        niche_pca[mask, 0],
        niche_pca[mask, 1],
        c=stage_colors[stage],
        label=stage,
        alpha=0.5,
        s=10,
    )
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Niche Embeddings (PCA)")
ax.legend()
plt.show()

## 5. Predict Cell State Transitions

Use the flow matching model to predict how cells transition between disease stages.

In [ ]:
# Predict Normal → Invasive transitions
predictions = model.predict(
    neighborhoods=neighborhoods,
    source_stage="Normal",
    target_stage="Invasive",
    num_integration_steps=8,
    return_trajectories=True,
)

print(f"Source embeddings: {predictions.source_embeddings.shape}")
print(f"Predicted embeddings: {predictions.predicted_embeddings.shape}")
print(f"Trajectories: {predictions.trajectories.shape if predictions.trajectories is not None else 'None'}")

In [ ]:
# Visualize transition vectors
velocities = predictions.predicted_embeddings - predictions.source_embeddings

# Project to 2D
combined = np.vstack([predictions.source_embeddings, predictions.predicted_embeddings])
pca = PCA(n_components=2)
combined_pca = pca.fit_transform(combined)

n = len(predictions.source_embeddings)
source_pca = combined_pca[:n]
target_pca = combined_pca[n:]

fig, ax = plt.subplots(figsize=(10, 8))

# Plot cells colored by stage
for stage in ["Normal", "Preinvasive", "Invasive"]:
    mask = adata.obs["stage"].values == stage
    ax.scatter(
        source_pca[mask, 0],
        source_pca[mask, 1],
        c=stage_colors[stage],
        label=stage,
        alpha=0.3,
        s=10,
    )

# Plot velocity arrows (subset for visibility)
idx = np.random.choice(n, min(200, n), replace=False)
ax.quiver(
    source_pca[idx, 0],
    source_pca[idx, 1],
    target_pca[idx, 0] - source_pca[idx, 0],
    target_pca[idx, 1] - source_pca[idx, 1],
    color="black",
    alpha=0.5,
    scale=15,
)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Predicted Transition Velocities (Normal → Invasive)")
ax.legend()
plt.show()

## 6. Analyze Attention Weights

Understand which spatial features the model attends to.

In [ ]:
if niche_output.attention_weights is not None:
    # Token names
    token_names = ["Receiver", "Ring 1", "Ring 2", "Ring 3", "Ring 4", "HLCA", "LuCA", "Pathway", "Stats"]
    n_tokens = niche_output.attention_weights.shape[1]
    token_names = token_names[:n_tokens]
    
    # Average attention by stage
    attention_by_stage = {}
    for stage in ["Normal", "Preinvasive", "Invasive"]:
        mask = adata.obs["stage"].values == stage
        attention_by_stage[stage] = niche_output.attention_weights[mask].mean(axis=0)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(token_names))
    width = 0.25
    
    for i, (stage, attn) in enumerate(attention_by_stage.items()):
        ax.bar(x + i * width, attn, width, label=stage, color=stage_colors[stage])
    
    ax.set_xlabel("Token")
    ax.set_ylabel("Attention Weight")
    ax.set_title("Niche Token Attention by Stage")
    ax.set_xticks(x + width)
    ax.set_xticklabels(token_names, rotation=45, ha="right")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No attention weights available (model may not use attention)")

## 7. Export Results

Save embeddings and predictions for downstream analysis.

In [ ]:
# Convert predictions to DataFrame
results_df = predictions.to_dataframe()
print(f"Results DataFrame: {results_df.shape}")
print(results_df.head())

In [ ]:
# Save to files
# results_df.to_parquet("predictions.parquet")
# adata.write("adata_with_embeddings.h5ad")

print("Done! Key results:")
print(f"  - Niche embeddings stored in adata.obsm['X_niche']")
print(f"  - Predictions available as DataFrame")

## Next Steps

- **Training**: See `02_training.ipynb` for training your own model
- **Biological Analysis**: See `03_biological_interpretation.ipynb` for analyzing ligand-receptor interactions and pathways
- **Advanced**: See the full documentation at [docs/](../docs/)